# Digital Image Processing Lab - Assignment 6
**Problem Statement:**
Write a program to smooth the input image using:
1. **Averaging filter (un-weighted)**
2. **Weighted filter** given by $h(x, y) = \max(|x|, |y|)$
3. **Gaussian filter**

*Note:* Input the filter size and other parameters such as $\sigma$ in case of Gaussian filter, method of padding (replication, zero padding, etc.) from the user.

---
### Theoretical Foundations
Smoothing filters suppress high-frequency components (noise and fine edges) using 2D spatial convolution:
$$g(x, y) = \sum_{s=-a}^a \sum_{t=-b}^b h(s, t) f(x + s, y + t)$$

1. **Averaging Filter (Box Filter):**
   $$h(x, y) = \frac{1}{K^2}$$
2. **Weighted Filter ($h(x, y) = \max(|x|, |y|)$):**
   Coordinates centered at $(0, 0)$ with $x, y \in [-\lfloor K/2 \rfloor, \lfloor K/2 \rfloor]$.
   Values grow radially outward with Chebyshev distance. Normalized by $\sum h(x, y)$.
3. **Gaussian Filter:**
   $$h(x, y) = \frac{1}{2\pi \sigma^2} e^{-\frac{x^2 + y^2}{2\sigma^2}}$$


In [ ]:
# Cell 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['image.cmap'] = 'gray'


In [ ]:
# Cell 2: Test Image & Noise Setup
def get_test_image():
    # Base pattern with geometric structures
    img = np.zeros((256, 256), dtype=np.float32)
    cv2.rectangle(img, (40, 40), (120, 120), 200, -1)
    cv2.circle(img, (180, 180), 50, 160, -1)
    cv2.putText(img, "SMOOTH", (30, 200), cv2.FONT_HERSHEY_SIMPLEX, 1.1, 240, 3)
    
    # Add Gaussian noise to demonstrate smoothing
    np.random.seed(42)
    noise = np.random.normal(0, 25, img.shape)
    noisy_img = np.clip(img + noise, 0, 255).astype(np.uint8)
    return noisy_img

noisy_image = get_test_image()
print("Noisy test image generated.")


In [ ]:
# Cell 3: Kernel Generators
def get_averaging_kernel(ksize):
    """Generates un-weighted averaging kernel of size ksize x ksize."""
    return np.ones((ksize, ksize), dtype=np.float64) / (ksize * ksize)

def get_max_weighted_kernel(ksize):
    """Generates weighted filter h(x,y) = max(|x|, |y|) centered at origin."""
    radius = ksize // 2
    y, x = np.mgrid[-radius:radius+1, -radius:radius+1]
    kernel = np.maximum(np.abs(x), np.abs(y)).astype(np.float64)
    total = np.sum(kernel)
    if total > 0:
        kernel = kernel / total
    return kernel

def get_gaussian_kernel(ksize, sigma):
    """Generates Gaussian kernel with parameter sigma."""
    radius = ksize // 2
    y, x = np.mgrid[-radius:radius+1, -radius:radius+1]
    kernel = np.exp(-(x**2 + y**2) / (2.0 * sigma**2))
    return kernel / np.sum(kernel)

print("Sample 3x3 Max-Weighted Kernel:")
print(get_max_weighted_kernel(3))


In [ ]:
# Cell 4: 2D Spatial Convolution with User Padding
def convolve2d_custom(image, kernel, pad_method='replicate'):
    """Performs 2D spatial convolution from scratch supporting multiple padding options."""
    k_h, k_w = kernel.shape
    pad_h, pad_w = k_h // 2, k_w // 2
    
    if pad_method == 'zero':
        padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant', constant_values=0)
    elif pad_method == 'replicate':
        padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='edge')
    elif pad_method == 'reflect':
        padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='reflect')
    else:
        raise ValueError(f"Unknown padding method: {pad_method}")
    
    # Fast convolution using sliding window views
    h, w = image.shape
    sub_matrices = np.lib.stride_tricks.sliding_window_view(padded, (k_h, k_w))
    filtered = np.einsum('ij,klij->kl', kernel, sub_matrices)
    
    return np.clip(np.round(filtered), 0, 255).astype(np.uint8)

# Colab User Inputs
filter_size = 5 #@param [3, 5, 7, 9] {type:"raw"}
gaussian_sigma = 1.5 #@param {type:"number"}
padding_type = "replicate" #@param ["replicate", "zero", "reflect"]

# Generate kernels
k_avg = get_averaging_kernel(filter_size)
k_max = get_max_weighted_kernel(filter_size)
k_gauss = get_gaussian_kernel(filter_size, gaussian_sigma)

# Perform smoothing
smooth_avg = convolve2d_custom(noisy_image, k_avg, pad_method=padding_type)
smooth_max = convolve2d_custom(noisy_image, k_max, pad_method=padding_type)
smooth_gauss = convolve2d_custom(noisy_image, k_gauss, pad_method=padding_type)

print(f"Smoothing completed using {filter_size}x{filter_size} kernels with {padding_type} padding.")


In [ ]:
# Cell 5: Visualizations & Comparison
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# Row 1: Images
axes[0, 0].imshow(noisy_image)
axes[0, 0].set_title("Noisy Input Image", fontweight='bold'); axes[0, 0].axis('off')

axes[0, 1].imshow(smooth_avg)
axes[0, 1].set_title(f"1. Averaging Filter ({filter_size}x{filter_size})", fontweight='bold'); axes[0, 1].axis('off')

axes[0, 2].imshow(smooth_max)
axes[0, 2].set_title(f"2. Weighted Filter h=max(|x|,|y|)", fontweight='bold'); axes[0, 2].axis('off')

axes[0, 3].imshow(smooth_gauss)
axes[0, 3].set_title(f"3. Gaussian Filter (sigma={gaussian_sigma})", fontweight='bold'); axes[0, 3].axis('off')

# Row 2: Kernel Heatmaps
axes[1, 0].axis('off')
axes[1, 0].text(0.1, 0.4, f"Filter Parameters:\n• Size: {filter_size}x{filter_size}\n• Sigma: {gaussian_sigma}\n• Padding: {padding_type}", 
                fontsize=13, family='monospace', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.6))

im1 = axes[1, 1].imshow(k_avg, cmap='viridis')
axes[1, 1].set_title("Averaging Kernel Weights", fontweight='bold')
plt.colorbar(im1, ax=axes[1, 1], fraction=0.046)

im2 = axes[1, 2].imshow(k_max, cmap='viridis')
axes[1, 2].set_title("Max-Weighted Kernel", fontweight='bold')
plt.colorbar(im2, ax=axes[1, 2], fraction=0.046)

im3 = axes[1, 3].imshow(k_gauss, cmap='viridis')
axes[1, 3].set_title("Gaussian Kernel Weights", fontweight='bold')
plt.colorbar(im3, ax=axes[1, 3], fraction=0.046)

plt.tight_layout()
plt.show()


### Performance Analysis
- **Averaging Filter:** Simple and fast, effectively removes high frequency noise but creates noticeable blurring around sharp boundaries.
- **Max-Weighted Filter ($h=\max(|x|, |y|)$):** Assigns heavier weight to outer peripheral pixels than the center.
- **Gaussian Filter:** Yields the most natural, isotropic blur without high-frequency ringing artifacts, preserving edge structures better than box averaging.
